In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [2]:
df = pd.read_csv("../raw_data/articles.csv")
df.drop(columns=["detail_desc"]).head(5)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear"
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear"


In [3]:
print(df['product_group_name'].unique())
product_group = 'Shoes'

['Garment Upper body' 'Underwear' 'Socks & Tights' 'Garment Lower body'
 'Accessories' 'Items' 'Nightwear' 'Unknown' 'Underwear/nightwear' 'Shoes'
 'Swimwear' 'Garment Full body' 'Cosmetic' 'Interior textile' 'Bags'
 'Furniture' 'Garment and Shoe care' 'Fun' 'Stationery']


In [4]:
print(f"The subcategories of {product_group} are: ")
df[df['product_group_name']==product_group]['product_type_name'].unique()

The subcategories of Shoes are: 


array(['Boots', 'Sneakers', 'Other shoe', 'Sandals', 'Slippers',
       'Ballerinas', 'Flat shoe', 'Wedge', 'Pumps', 'Flip flop', 'Bootie',
       'Heeled sandals', 'Flat shoes', 'Heels', 'Moccasins',
       'Pre-walkers'], dtype=object)

In [5]:
df_filtered = df[df['product_group_name']==product_group]
df_filtered.head(3)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
124,181160009,181160,Eva chelsea boot,87,Boots,Shoes,1010016,Solid,17,Yellowish Brown,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Chelsea boots with elasticated gores in the si...
257,212042036,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,1,Other,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...
258,212042043,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,9,Black,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...


In [6]:
df_filtered['index_group_name'].unique()

array(['Divided', 'Menswear', 'Ladieswear', 'Baby/Children', 'Sport'],
      dtype=object)

In [7]:
df_trans = pd.read_csv("../raw_data/transactions_train.csv")
df_trans['t_dat']= pd.to_datetime(df_trans['t_dat'])
df_trans.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [8]:
df_trans_filtered = df_trans[df_trans['article_id'].isin(df_filtered['article_id'])]

In [9]:
print("Saving filtered dataFrames")

Saving filtered dataFrames


In [10]:
df_filtered.to_csv("../raw_data/articles_filtered.csv")

In [11]:
df_trans_filtered.to_csv("../raw_data/transaction_filtered.csv")

In [12]:
!ls /kaggle/working

ls: /kaggle/working: No such file or directory


# Image Filter

In [13]:
import os
import shutil
from pathlib import Path

In [14]:
needed_article_ids = df_filtered['article_id'].unique()
needed_article_ids.shape

(5283,)

In [15]:
source_images_path = Path('../raw_data/images_256_256')
dest_images_path = Path('../raw_data/images_filtered')

In [16]:
copied_count = 0
missing_count = 0

for article_id in needed_article_ids:
    article_str = str(article_id).zfill(10)
    subfolder = article_str[:3]

    source_subfolder = source_images_path / subfolder
    dest_subfolder = dest_images_path / subfolder
    source_file = source_subfolder / f"{article_str}.jpg"
    dest_file = dest_subfolder / f"{article_str}.jpg"

    if source_file.exists():
        dest_subfolder.mkdir(exist_ok=True)

        shutil.copy2(source_file, dest_file)
        copied_count += 1
    else:
        missing_count += 1

print(f"Copied {copied_count} images")
print(f"Missing images: {missing_count}")
print(f"Total article IDs: {needed_article_ids.shape}")

Copied 5156 images
Missing images: 127
Total article IDs: (5283,)


In [18]:
list(df_filtered)

['article_id',
 'product_code',
 'prod_name',
 'product_type_no',
 'product_type_name',
 'product_group_name',
 'graphical_appearance_no',
 'graphical_appearance_name',
 'colour_group_code',
 'colour_group_name',
 'perceived_colour_value_id',
 'perceived_colour_value_name',
 'perceived_colour_master_id',
 'perceived_colour_master_name',
 'department_no',
 'department_name',
 'index_code',
 'index_name',
 'index_group_no',
 'index_group_name',
 'section_no',
 'section_name',
 'garment_group_no',
 'garment_group_name',
 'detail_desc']

# Deviding functions: 

In [ ]:

def load_and_prepare_data(num_images=100):
    """
    Step 1: Load images and labels from CSV
    Returns:
        X (numpy array): Image data
        y (numpy array): One-hot encoded labels
        categories (list): List of category names
    """
    print(f"Step 1: Loading {num_images} images...")

    # Load CSV
    df = pd.read_csv('../raw_data/articles_filtered.csv')
    df = df.head(num_images)
    print(f"Loaded {len(df)} articles from CSV")

    categories = sorted(df['product_type_name'].unique())
    print(f"Categories ({len(categories)}): {categories}")

    # Create category to index mapping
    category_to_idx = {cat: idx for idx, cat in enumerate(categories)}

    images = []
    labels = []

    # Load all images
    for idx, row in df.iterrows():
        article_id = str(row['article_id']).zfill(10)
        subfolder = article_id[:3]
        category = row['product_type_name']

        # Build image path
        image_path = Path('../raw_data/images_filtered') / subfolder / f"{article_id}.jpg"

        # Skip if image doesn't exist
        if not image_path.exists():
            continue

        try:
            # Load and preprocess image
            img = Image.open(image_path).convert('RGB')
            img = img.resize((256, 256), Image.LANCZOS)
            img_array = np.array(img, dtype=np.float32) / 255.0

            images.append(img_array)
            labels.append(category_to_idx[category])
        except Exception as e:
            print(f"Skipping {article_id}: {e}")
            continue

    # Convert to numpy arrays
    X = np.array(images)
    y = np.array(labels)
    y = to_categorical(y, num_classes=len(categories))

    print(f"✅ Loaded {len(images)} images")
    print(f"   X shape: {X.shape}")
    print(f"   y shape: {y.shape}")

    return X, y, categories


In [35]:
X, y, categories = load_and_prepare_data(100)

Step 1: Loading 100 images...
Loaded 100 articles from CSV
Categories (7): ['Ballerinas', 'Boots', 'Flat shoe', 'Other shoe', 'Sandals', 'Slippers', 'Sneakers']
✅ Loaded 90 images
   X shape: (90, 256, 256, 3)
   y shape: (90, 7)


In [ ]:
def split_data(X, y, split_ratio=0.8):
    """
    Step 2: Split data into training and validation sets

    Args:
        X: Image data
        y: Labels
        split_ratio: Ratio for train/validation split (default 0.8 = 80/20)

    Returns:
        X_train, y_train, X_val, y_val
    """
    print(f"\nStep 2: Splitting data ({int(split_ratio*100)}/{int((1-split_ratio)*100)})...")

    split_idx = int(split_ratio * len(X))
    X_train = X[:split_idx]
    y_train = y[:split_idx]
    X_val = X[split_idx:]
    y_val = y[split_idx:]

    print(f"✅ Training set: {X_train.shape[0]} images")
    print(f"   Validation set: {X_val.shape[0]} images")

    return X_train, y_train, X_val, y_val


In [ ]:
def build_model(num_categories):
    """
    Step 3: Build and compile the CNN model
    """
    print(f"\nStep 3: Building model for {num_categories} categories...")

    model = Sequential()
    model.add(layers.Conv2D(16, (4, 4), padding='same', activation='relu', input_shape=(256, 256, 3)))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(16, (3, 3), padding='same', activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(num_categories, activation='softmax'))

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    print("✅ Model built and compiled!")
    return model


In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=10):
    """
    Step 4: Train the model
    """
    print(f"\nStep 4: Training model for {epochs} epochs...")

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=32,
        verbose=1
    )

    print("✅ Training complete!")
    return model, history


In [ ]:
def predict_category(model, image_path, categories):
    """
    Step 5: Predict category for a new image
    """
    print(f"\nStep 5: Predicting category for: {image_path}")

    # Load and preprocess image
    if isinstance(image_path, str):
        img = Image.open(image_path).convert('RGB')
        img = img.resize((256, 256), Image.LANCZOS)
        img_array = np.array(img, dtype=np.float32) / 255.0
    else:
        img_array = image_path

    # Add batch dimension
    if len(img_array.shape) == 3:
        img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(predictions[0])
    predicted_category = categories[predicted_idx]

    print(f"✅ Predicted: {predicted_category}")
    return predicted_category


In [41]:
X_train, y_train, X_val, y_val = split_data(X, y)

model = build_model(len(categories))

model, history = train_model(model, X_train, y_train, X_val, y_val, epochs=10)


Step 2: Splitting data (80/19)...
✅ Training set: 72 images
   Validation set: 18 images

Step 3: Building model for 7 categories...
✅ Model built and compiled!

Step 4: Training model for 10 epochs...
Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 371ms/step - accuracy: 0.2222 - loss: 1.8727 - val_accuracy: 0.4444 - val_loss: 1.7223
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 283ms/step - accuracy: 0.3056 - loss: 1.7322 - val_accuracy: 0.3333 - val_loss: 1.8312
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 302ms/step - accuracy: 0.2778 - loss: 1.6823 - val_accuracy: 0.3333 - val_loss: 1.8448
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 298ms/step - accuracy: 0.2917 - loss: 1.6521 - val_accuracy: 0.3333 - val_loss: 1.7673
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 308ms/step - accuracy: 0.2917 - loss: 1.6295 - val_accuracy: 0.6111 - val_loss: 1.7865
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 293ms/step - accuracy: 0.3056 - loss: 1.5909 - val_accuracy: 0.4444 - val_loss: 1.8491
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 26

In [42]:
import requests
from PIL import Image
from io import BytesIO

url = "https://cdnimg.vivaia.com/vivaia/products/1684736989-5d54aff0-add6-4ec2-9bb5-d21184b5af62.jpg?imresize=800x800"
response = requests.get(url)
img = Image.open(BytesIO(response.content))
img.save("test_shoe.jpg")

# Now predict
result = predict_category(model, "test_shoe.jpg", categories)
print(f"Result: {result}")


Step 5: Predicting category for: test_shoe.jpg
✅ Predicted: Sneakers
Result: Sneakers


In [ ]:
# Full pipeline
def classify_subcategory(image_path, num_images=100, epochs=10):
    """
    Complete pipeline: All steps in one function
    """

    # Step 1: Load data
    X, y, categories = load_and_prepare_data(num_images)

    # Step 2: Split data
    X_train, y_train, X_val, y_val = split_data(X, y)

    # Step 3: Build model
    model = build_model(len(categories))

    # Step 4: Train
    model, history = train_model(model, X_train, y_train, X_val, y_val, epochs)

    # Step 5: Predict
    result = predict_category(model, image_path, categories)

    print(f"Result: {result}")

    return result
